# Домашнее задание: Где дешевле жить? Предсказание цен в Airbnb
# Занятие: Ансамбли моделей

# Этап 0. Подготовка окружения и импорты

In [ ]:
import os
from itertools import combinations

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from lightgbm import LGBMRegressor
from sklearn.linear_model import ElasticNetCV, LassoCV, LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Константы
RANDOM_STATE = 42
TEST_SIZE = 0.3
MANHATTAN_CENTER = (40.7831, -73.9712)

# Настройки отображения
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
sns.set_style("whitegrid")

# Этап 1. Загрузка данных

In [ ]:
path = kagglehub.dataset_download("dgomonov/new-york-city-airbnb-open-data")
file_path = os.path.join(path, "AB_NYC_2019.csv")
df = pd.read_csv(file_path)
df.info()

In [ ]:
df.head()

# Этап 2. EDA и предобработка данных

In [ ]:
# 2.1 Обработка пропусков
df["host_name"] = df["host_name"].fillna("-")
print(f"Пустые значения 'host_name' заполнены '-'")

df["neighbourhood"] = df["neighbourhood"].fillna("-")
print(f"Пустые значения 'neighbourhood' заполнены '-'")

df["reviews_per_month"] = df["reviews_per_month"].fillna(0)
print(f"Пустые значения 'reviews_per_month' заполнены '0'")

In [ ]:
df.info()

In [ ]:
# 2.2 Создание признаков из last_review
df["last_review"] = pd.to_datetime(df["last_review"], errors="coerce")
print(f"Значения 'last_review' преобразованы в формат даты")

df["has_reviews"] = df["last_review"].notna().astype(int)
print(f"Создан 'has_reviews' из значений 'has_reviews', только '0' и '1'")
print(f"Всего {df['has_reviews'].sum()} записей с отзывами")

df["days_since_review"] = (pd.Timestamp.now() - df["last_review"]).dt.days
print(f"Создан 'days_since_review' как разница текущей даты и 'last_review'")

df["days_since_review"] = df["days_since_review"].fillna(-1)
print(f"Пустые значения 'days_since_review' заполнены '-1'")

df = df.drop("last_review", axis=1)
print(f"Удален признак 'last_review'")

In [ ]:
df.info()

In [ ]:
# 2.3 One-Hot Encoding категориальных переменных
df = pd.get_dummies(
    df,
    columns=["neighbourhood_group", "room_type"],
    drop_first=False,
    dtype=int
)

print(f"Признак 'neighbourhood_group' разделен 5 колонок")
print(f"Признак 'room_type' разделен на 3 колонки")

In [ ]:
df.info()

In [ ]:
# 2.4 Бининг minimum_nights
def bin_and_encode(df, column, bins, labels, prefix=None):
    """Разбивает числовой признак на бины и кодирует в One-Hot"""
    bin_col = f"{column}_bin"
    
    df[bin_col] = pd.cut(
        df[column],
        bins=bins,
        labels=labels,
        right=True,
        include_lowest=True
    )

    df = pd.get_dummies(
        df,
        columns=[bin_col],
        prefix=prefix or column,
        dtype=int
    )

    print(f"+ {column} -> {df.filter(like=prefix or column).columns.tolist()}")
    return df

In [ ]:
bins = [0, 1, 2, 3, 6, np.inf]

labels = ["1 night", "2 nights", "3 nights", "4-6 nights", "1 week+"]

df = bin_and_encode(
    df,
    column="minimum_nights",
    bins=bins,
    labels=labels,
    prefix="min_nights"
)

In [ ]:
df.info()

In [ ]:
# 2.5 Создание новых признаков

# Лог-трансформация количества listings у хоста
df["host_listings_log"] = np.log1p(df["calculated_host_listings_count"])

# Флаг профессионального хоста
df["is_professional_host"] = (df["calculated_host_listings_count"] >= 10).astype(int)
print(f"Создан 'host_listings_log' создан")
print(f"Создан 'is_professional_host' создан: {df['is_professional_host'].sum()} крупных владельцев")

In [ ]:
# Расстояние до Манхэттена
df["distance_to_manhattan"] = np.sqrt(
    (df["latitude"] - MANHATTAN_CENTER[0]) ** 2
    + 
    (df["longitude"] - MANHATTAN_CENTER[1]) ** 2
)
print(f"Создан 'distance_to_manhattan': диапазон [{df['distance_to_manhattan'].min():.2f}, {df['distance_to_manhattan'].max():.2f}]")

In [ ]:
# 2.6 Удаление ненужных колонок
cols_to_drop = [
    "id",
    "name",
    "host_name",
    "host_id",
    "minimum_nights",
    "calculated_host_listings_count",
]

df = df.drop(columns=cols_to_drop, errors="ignore")
print(f"Удалено колонок: {len(cols_to_drop)}")
print(f"Осталось признаков: {len(df.columns)}")

In [ ]:
df.info()

In [ ]:
df.head()

# Этап 3. Анализ и очистка целевой переменной

In [ ]:
print(f"Статистика price до обработки:")
print(df["price"].describe())

In [ ]:
# Визуализация распределения
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x="price", kde=True)
plt.title("Распределение цен (price) до обработки")
plt.xlabel("Цена ($)")
plt.tight_layout()
plt.show()

In [ ]:
# Анализ выбросов
print(f"Записей с price=0: {(df['price'] == 0).sum()}")
print(f"Записей с price>1000: {(df['price'] > 1000).sum()}")
print(f"Записей с price>5000: {(df['price'] > 5000).sum()}")

#### Очистка целевой переменной

In [ ]:
# Удаляем нулевые цены
df = df[df["price"] > 0].copy()
print(f"Удалены записи с price '0'")

In [ ]:
# Обрезаем выбросы по 99.5% перцентилю
upper_limit = df["price"].quantile(0.995)
print(f"Верхний порог (99.5%): ${upper_limit:.0f}")
df["price"] = df["price"].clip(upper=upper_limit)

In [ ]:
print(f"Статистика price после обработки:")
print(df["price"].describe().round(2))

In [ ]:
# Визуализация после обработки
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x="price", kde=True)
plt.title("Распределение цен (price) после обработки")
plt.xlabel("Цена ($)")
plt.tight_layout()
plt.show()

# Этап 4. EDA - корреляционный анализ

In [ ]:
# Список числовых колонок для анализа
one_hot_cols = [
    "neighbourhood_group_Bronx",
    "neighbourhood_group_Brooklyn",
    "neighbourhood_group_Manhattan",
    "neighbourhood_group_Queens",
    "neighbourhood_group_Staten Island",
    "room_type_Entire home/apt",
    "room_type_Private room",
    "room_type_Shared room",
    "min_nights_1 night",
    "min_nights_2 nights",
    "min_nights_3 nights",
    "min_nights_4-6 nights",
    "min_nights_1 week+",
]

all_numeric = [col for col in df.columns if df[col].dtype in ["float64", "int64"]]

numeric_cols = [
    col
    for col in all_numeric
    if col not in one_hot_cols
       and col != "price"
       and col not in ["has_reviews", "is_professional_host"]
]

numeric_cols.append("price")

print(f"Колонки для корреляции: {len(numeric_cols)}")
numeric_cols

In [ ]:
# Матрица корреляций
corr_matrix = df[numeric_cols].corr()

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=False, cmap="coolwarm", center=0, linewidths=0.5, square=True)
plt.title("Матрица корреляций числовых признаков с price")
plt.tight_layout()
plt.show()

In [ ]:
# Корреляции с target
price_corr = corr_matrix["price"].drop("price").sort_values(ascending=False)

In [ ]:
print("Корреляция признаков с price:")
for feat, val in price_corr.items():
    marker = "v" if abs(val) > 0.3 else " "
    print(f"{marker} {feat:25s}: {val:+.3f}")

In [ ]:
# Топ коррелирующих пар
def get_correlation_pairs_df(corr_matrix):
    data = [
        (min(f1, f2), max(f1, f2), corr_matrix.loc[f1, f2])
        for f1, f2 in combinations(corr_matrix.columns, 2)
    ]
    corr_df = pd.DataFrame(data, columns=["feature1", "feature2", "correlation"])
    return corr_df.sort_values(by="correlation", key=abs, ascending=False).reset_index(drop=True)

In [ ]:
corr_pairs = get_correlation_pairs_df(corr_matrix)

In [ ]:
print("Топ-10 пар по модулю корреляции:")
print(corr_pairs.head(10))

# Этап 5. Подготовка данных для моделирования

In [ ]:
# Список признаков для модели
features = [
    # Район
    "neighbourhood_group_Bronx",
    "neighbourhood_group_Brooklyn",
    "neighbourhood_group_Manhattan",
    "neighbourhood_group_Queens",
    "neighbourhood_group_Staten Island",
    
    # География (оставляем distance_to_manhattan, убираем lat/lon)
    "distance_to_manhattan",
    
    # Тип жилья
    "room_type_Entire home/apt",
    "room_type_Private room",
    "room_type_Shared room",
    
    # Минимальное пребывание
    "min_nights_1 night",
    "min_nights_2 nights",
    "min_nights_3 nights",
    "min_nights_4-6 nights",
    "min_nights_1 week+",
    
    # Отзывы
    "number_of_reviews",
    "reviews_per_month",
    "has_reviews",
    "days_since_review",
    
    # Владелец
    "host_listings_log",
    "is_professional_host",
    
    # Доступность
    "availability_365",
]

print(f"Признаков: {len(features)}")
print(f"Записей: {len(df[features])}")

In [ ]:
X = df[features]
y = df["price"]

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape[0]} записей")
print(f"Test: {X_test.shape[0]} записей")

In [ ]:
# Масштабирование признаков
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Признаки масштабированы: StandardScaler")

### Лог-трансформация целевой переменной

In [ ]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(f"Target трансформирован: log1p")

# Этап 6. Моделирование

In [ ]:
# Словарь моделей
models = {
    "Linear": LinearRegression(),
    "RidgeCV": RidgeCV(alphas=[0.01, 0.1, 1, 10, 100], cv=3),
    "LassoCV": LassoCV(alphas=[0.01, 0.1, 1, 10], cv=3, max_iter=10000),
    "ElasticNetCV": ElasticNetCV(alphas=[0.01, 0.1, 1], l1_ratio=[0.3, 0.5, 0.7], cv=3, max_iter=10000),
}

In [ ]:
# Результаты
results = []

# Обучение линейных моделей
for name, model in models.items():
    model.fit(X_train_scaled, y_train_log)

    # Предсказания
    y_pred_log = model.predict(X_test_scaled)
    y_pred = np.expm1(y_pred_log)

    # Метрики
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({"model": name, "MAE": mae, "RMSE": rmse, "R2": r2})

    print(f"Модель {name}, MAE: ${mae:.2f}, RMSE: ${rmse:.2f}, R²: {r2:.3f}")

In [ ]:
# LightGBM (для сравнения, без масштабирования)
lgb_model = LGBMRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)

lgb_model.fit(X_train, y_train_log)
y_pred_log = lgb_model.predict(X_test)
y_pred_lgb = np.expm1(y_pred_log)

results.append(
    {
        "model": "LightGBM",
        "MAE": mean_absolute_error(y_test, y_pred_lgb),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_lgb)),
        "R2": r2_score(y_test, y_pred_lgb),
    }
)

print(f"Модель {results[-1]['model']}, MAE: ${results[-1]['MAE']:.2f}, RMSE: ${results[-1]['RMSE']:.2f}, R²: {results[-1]['R2']:.3f}")

In [ ]:
# Итоговая таблица
results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
print(results_df.to_string(index=False))

# Этап 7. Визуализация результатов

In [ ]:
# 7.1 Actual vs Predicted для лучшей модели
best_model_name = results_df.iloc[0]["model"]
best_model = models.get(best_model_name, lgb_model if best_model_name == "LightGBM" else None)

if best_model_name == "LightGBM":
    y_pred_best = y_pred_lgb
else:
    y_pred_log = best_model.predict(X_test_scaled)
    y_pred_best = np.expm1(y_pred_log)

In [ ]:
# График Actual vs Predicted"
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_best, alpha=0.2, s=5, edgecolor="none")
plt.plot([0, 500], [0, 500], "r--", lw=2, label="Идеальное предсказание")
plt.xlabel("Фактическая цена ($)")
plt.ylabel("Предсказанная цена ($)")
plt.title(f"{best_model_name}: Actual vs Predicted\nR² = {r2_score(y_test, y_pred_best):.3f}, MAE = ${mean_absolute_error(y_test, y_pred_best):.2f}")
plt.xlim(0, 500)
plt.ylim(0, 500)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 7.2 Feature Importance для Ridge
if "RidgeCV" in models:
    best_ridge = models["RidgeCV"]
    coefs = pd.Series(best_ridge.coef_, index=features)
    top_positive = coefs.sort_values(ascending=False).head(10)
    top_negative = coefs.sort_values().head(10)

    plt.figure(figsize=(10, 6))
    pd.concat([top_negative, top_positive]).plot(kind="barh")
    plt.axvline(0, color="gray", lw=0.5)
    plt.title("Важность признаков (коэффициенты Ridge)")
    plt.xlabel("Значение коэффициента (на лог-шкале цены)")
    plt.tight_layout()
    plt.show()

    print("\nТоп-5 положительных коэффициентов:")
    print(top_positive.head())
    
    print("\nТоп-5 отрицательных коэффициентов:")
    print(top_negative.head())

# Этап 8. Сравнение с baseline (без предобработки)

In [ ]:
# Вариант А: без предобработки
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

model_raw = LinearRegression()
model_raw.fit(X_train_raw, y_train_raw)
pred_raw = model_raw.predict(X_test_raw)

r2_raw = r2_score(y_test_raw, pred_raw)
mae_raw = mean_absolute_error(y_test_raw, pred_raw)

print(f"Baseline (без предобработки): MAE = ${mae_raw:.2f}, R² = {r2_raw:.3f}")

In [ ]:
# Вариант Б: с предобработкой
r2_improved = r2_score(y_test, y_pred_best)
mae_improved = mean_absolute_error(y_test, y_pred_best)

print(f"Improved (log + scaling): MAE = ${mae_improved:.2f}, R² = {r2_improved:.3f}")

In [ ]:
# Итог
print(f"Улучшение MAE: ${mae_raw - mae_improved:+.2f}")
print(f"Улучшение R²: {r2_improved - r2_raw:+.3f}")

# Этап 9. Итоговые выводы

## 1. Опробованные техники

| Техника                                 | Помогло? | Комментария                                      |
| --------------------------------------- | -------- | ------------------------------------------------ |
| Удаление id, name, host_*               | +        | Убрали шум                                       |
| One-Hot Encoding категорий              | +        | Обязательно для линейных моделей                 |
| Бининг minimum_nights                   | 0        | Улучшил интерпретируемость, малый прирост метрик |
| Создание has_reviews, days_since_review | +        | Добавили сигнал о наличии отзывов                |
| log1p(price) + StandardScaler           | ++       | Ключевое улучшение                               |
| Создание distance_to_manhattan          | +        | Лучший гео-признак вместо lat/lon                |
| Удаление latitude/longitude             | +        | Снизил риск overfitting                          |


## 2. Итоговые метрики

| model        | MAE       | RMSE       | R2       |
| ------------ | --------- | ---------- | -------- |
| LightGBM     | 51.863056 | 106.395376 | 0.329349 |
| Linear       | 55.417820 | 111.922076 | 0.257865 |
| RidgeCV      | 55.412239 | 111.945436 | 0.257555 |
| ElasticNetCV | 55.373091 | 112.233326 | 0.253732 |
| LassoCV      | 55.446882 | 112.783102 | 0.246403 |

## 3. Ключевые выводы

1. **Лог-трансформация целевой переменной** — важный шаг для задач с ценами, снижает влияние выбросов.

2. **RidgeCV** - лучший среди линейных моделей.

3. **LightGBM** - лучше линейных моделей, но требует больше настройки.

4. **neighbourhood_group_Manhattan** - самый влиятельный признак, Манхэттен — самый дорогой район в Нью-Йорке.

5. **distance_to_manhattan** - работает лучше, более интерпретируемый гео-признак чем latitude/longitude.

6. **Крупные владельцы** - в среднем сдают дороже.